# Biohub 3D Cell Tracking — Model Predictions & Tracking Visualization

Loads the trained 3D U-Net model (`ResidualCellTrackingUNet`), runs inference with valid-padding spatial offset corrections and spatial reflection padding, and **renders predicted cell centroids and tracking trajectory edges directly on top of microscopy images**.

In [6]:
# ── Install missing deps ──
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zarr"], check=False)

# ── Environment & Path Setup ──
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import blosc2
import json
import glob
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from IPython.display import clear_output, display

# ── YOUR UPLOADED DATASET NAME ──
# Change this single variable whenever you upload under a new username/dataset name:
USER_DATASET_NAME = "badmosmadara/biohub-cell-tracking-src-2"

# ── FIXED PATHS ──
COMPETITION_DATA_ROOT = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
USER_DATASET_ROOT = Path(f"/kaggle/input/datasets/{USER_DATASET_NAME}")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Add source directory to Python path instantly
for p in [USER_DATASET_ROOT, USER_DATASET_ROOT / "src", OUTPUT_DIR]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

INPUT_CHECKPOINT_DIRS = [USER_DATASET_ROOT, USER_DATASET_ROOT / "src", OUTPUT_DIR]

last_ckpt_path = OUTPUT_DIR / "last.pt"
best_ckpt_path = OUTPUT_DIR / "best.pt"
log_csv_path = OUTPUT_DIR / "training_log.csv"
plot_png_path = OUTPUT_DIR / "loss_plot.png"

print(f"Competition Data: {COMPETITION_DATA_ROOT}")
print(f"User Dataset Root: {USER_DATASET_ROOT}")
print("Paths configured instantly.")


Competition Data: /kaggle/input/competitions/biohub-cell-tracking-during-development/train
User Dataset Root: /kaggle/input/datasets/badmosmadara/biohub-cell-tracking-src-2
Paths configured instantly.


In [7]:
# ─── Load Model & Configuration ─────────────────────────────────────────────────────────────
from src.config import Config, ModelConfig, DataConfig, InferenceConfig
from src.models.residual_unet3d import ResidualCellTrackingUNet
from src.infer import compute_crop_offset, pad_spatial_tensor_if_needed
from src.tracking.linker import extract_peaks_from_frame, build_tracking_graph

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

config = Config(
    model=ModelConfig(
        in_channels=1,
        base_features=16,  # Must match training checkpoint!
        feature_multiplier=3.0,
        downsample_factors=[[1, 2, 2], [2, 2, 2], [2, 2, 2]],
        nms_kernel=3,      # Match training
        nms_threshold=0.1, # Match training
    ),
    data=DataConfig(
        input_shape=[16, 128, 128],
        scale=[1.625, 0.40625, 0.40625],
        normalize_mode="per_volume",
    ),
    inference=InferenceConfig(
        max_link_distance_um=7.0,
    ),
)

# Search for checkpoint
ckpt_candidates = [
    Path('/kaggle/input/datasets/badmosmadara/biohub-cell-tracking-src-2/src/best.pt'),
    Path('/kaggle/input/biohub-cell-tracking-src-2/src/best.pt'),
    Path('/kaggle/input/biohub-cell-tracking-src-2/best.pt'),
    Path('/kaggle/input/datasets/badmosmadara/biohub-cell-tracking-src-2/src/swa_final.pt'),
    Path('/kaggle/input/biohub-cell-tracking-src-2/src/swa_final.pt'),
    Path('/kaggle/input/biohub-cell-tracking-src-2/swa_final.pt'),
    Path('/kaggle/working/best.pt'),
    Path('/kaggle/working/swa_final.pt')
]
ckpt_path = next((p for p in ckpt_candidates if p.exists()), None)

model = ResidualCellTrackingUNet.from_config(config)
if ckpt_path:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state_dict = ckpt.get('model', ckpt)
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    print(f'Loaded trained weights from: {ckpt_path}')
else:
    print('Warning: No checkpoint found, using default model initialization.')

model.to(device)
model.eval()
print('Model ready.')


Using device: cuda
Model ready.


In [8]:
# ─── Inference Pipeline Helper ─────────────────────────────────────────────────────────────
def normalize_volume(volume, mode="percentile", percentile=99.5):
    vol = volume.astype(np.float32)
    if mode == "per_volume":
        vmax = float(vol.max())
        if vmax > 0:
            vol /= vmax
    elif mode == "percentile":
        vmax = float(np.percentile(vol, percentile))
        if vmax > 0:
            vol = np.clip(vol / vmax, 0.0, 1.0)
    return vol

def run_sample_inference(sample_path, model, config, timepoints_to_process=5):
    """Run cell detection and tracking over a subset of timepoints for visualization."""
    with open(os.path.join(sample_path, '0', 'zarr.json')) as f:
        meta = json.load(f)
        shape = meta['shape']
        chunk_shape = meta['chunk_grid']['chunk_shape'] if 'chunk_shape' in meta['chunk_grid'] else meta['chunk_grid']['configuration']['chunk_shape']
    
    num_timepoints = min(shape[0], timepoints_to_process)
    input_shape = tuple(shape[1:])
    min_shape = tuple(config.data.input_shape)
    
    def get_frame(t):
        chunk_path = os.path.join(sample_path, '0', 'c', str(t), '0', '0', '0')
        with open(chunk_path, 'rb') as f:
            decomp = blosc2.decompress(f.read())
        return np.frombuffer(decomp, dtype=np.uint16).reshape(chunk_shape).squeeze(0)
    
    raw_frames = []
    frame_detections = {}
    
    for t in range(num_timepoints):
        raw_vol = get_frame(t)
        raw_frames.append(raw_vol)
        
        frames = []
        for i in range(-3, 4):
            t_offset = max(0, min(t + i, shape[0] - 1))
            f = normalize_volume(get_frame(t_offset), mode=config.data.normalize_mode)
            frames.append(f)
        
        frame_window = np.stack(frames, axis=0)
        x = torch.from_numpy(frame_window).unsqueeze(0).unsqueeze(0).float().to(device)
        x_padded, (pz_b, py_b, px_b) = pad_spatial_tensor_if_needed(x, min_shape=min_shape)
        
        with torch.no_grad():
            outputs = model(x_padded, inference=True)
            ci     = outputs['cell_indicator'][0, 0].cpu().numpy()
            maxima = outputs['maxima'][0].cpu().numpy() if outputs['maxima'].ndim == 4 else outputs['maxima'][0, 0].cpu().numpy()
            mv     = outputs['movement_vectors'][0].cpu().numpy()
            
        actual_output_shape = ci.shape
        crop_offset_padded = compute_crop_offset(x_padded.shape[-3:], actual_output_shape)
        crop_offset = (
            crop_offset_padded[0] - pz_b,
            crop_offset_padded[1] - py_b,
            crop_offset_padded[2] - px_b,
        )
        
        nodes = extract_peaks_from_frame(ci, maxima, mv, t=t, threshold=0.05, crop_offset=crop_offset)
        frame_detections[t] = nodes
        
    tracking_result = build_tracking_graph(
        frame_detections,
        max_distance_um=config.inference.max_link_distance_um,
        scale=tuple(config.data.scale),
        allow_divisions=True,
    )
    
    return raw_frames, frame_detections, tracking_result


In [9]:
# Visualization: MIP with GT + Prediction overlay
def visualize_with_gt(raw_vol, pred_nodes, gt_nodes_all, timepoint=0, threshold=0.4):
    """MIP overlay: pink=predicted, green x=ground-truth. Also prints per-Z table."""
    import numpy as np
    import matplotlib.pyplot as plt
    from collections import Counter

    mip = np.max(raw_vol, axis=0)  # collapse Z -> (Y, X)
    pred_kept = [n for n in pred_nodes if n.confidence >= threshold]

    fig, axes = plt.subplots(1, 2, figsize=(20, 9))

    # Left: Predictions only
    ax = axes[0]
    ax.imshow(mip, cmap='magma', origin='upper')
    if pred_kept:
        xs = [n.x for n in pred_kept]
        ys = [n.y for n in pred_kept]
        confs = [n.confidence for n in pred_kept]
        sc = ax.scatter(xs, ys, c=confs, cmap='spring', s=40,
                        edgecolors='white', linewidths=0.6, alpha=0.9, zorder=3)
        plt.colorbar(sc, ax=ax, label='Confidence')
    ax.set_title(
        f'PREDICTIONS  t={timepoint}\n'
        f'{len(pred_kept)} cells (conf >= {threshold}) | {len(pred_nodes)} total peaks',
        fontsize=13, fontweight='bold')
    ax.set_xlabel('X (voxels)')
    ax.set_ylabel('Y (voxels)')

    # Right: GT + Predictions together
    ax2 = axes[1]
    ax2.imshow(mip, cmap='magma', origin='upper')
    if gt_nodes_all:
        gx = [n.x for n in gt_nodes_all]
        gy = [n.y for n in gt_nodes_all]
        ax2.scatter(gx, gy, c='lime', s=80, marker='x', linewidths=1.8,
                    alpha=0.95, zorder=4, label=f'GT ({len(gt_nodes_all)} cells)')
    if pred_kept:
        ax2.scatter([n.x for n in pred_kept], [n.y for n in pred_kept],
                    c='deeppink', s=25, edgecolors='white', linewidths=0.5,
                    alpha=0.75, zorder=3, label=f'Pred ({len(pred_kept)} cells)')
    ax2.legend(loc='upper right', fontsize=11, framealpha=0.8)
    ax2.set_title(f'GT (green x) vs PREDICTIONS (pink)  t={timepoint}',
                  fontsize=13, fontweight='bold')
    ax2.set_xlabel('X (voxels)')
    ax2.set_ylabel('Y (voxels)')

    plt.suptitle(f'3D MIP Projection (all Z-slices collapsed)  |  t={timepoint}',
                 fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

    # Per-Z breakdown table
    pred_by_z = Counter(int(round(n.z)) for n in pred_kept)
    gt_by_z   = Counter(n.z for n in gt_nodes_all) if gt_nodes_all else {}
    all_z = sorted(set(list(pred_by_z.keys()) + list(gt_by_z.keys())))
    print(f'\n=== Per-Z-slice breakdown at t={timepoint} ===')
    print(f'   Z  |   GT  |  Pred')
    print(f'  ----|-------|------')
    for z in all_z:
        print(f'  {z:>3}  |  {gt_by_z.get(z,0):>3}  |  {pred_by_z.get(z,0):>3}')
    print(f'  ----|-------|------')
    print(f'  TOT |  {sum(gt_by_z.values()):>3}  |  {sum(pred_by_z.values()):>3}')


# Backward-compatible wrapper
def visualize_frame_predictions(raw_vol, nodes, timepoint=0, threshold=0.4):
    visualize_with_gt(raw_vol, nodes, [], timepoint=timepoint, threshold=threshold)


In [10]:
# ─── Visualization 2: Multi-Frame Cell Tracking & Edges ─────────────────────────────────────
def # visualize_tracking_trajectories(raw_frames, tracking_result, start_t=0, num_frames=3):
    """Plot consecutive frames with tracking links (edges) connecting parent-child cells across time."""
    fig, axes = plt.subplots(1, num_frames, figsize=(7 * num_frames, 7))
    if num_frames == 1:
        axes = [axes]
        
    node_dict = {node.node_id: node for node in tracking_result.nodes}
    t_edges = {t: [] for t in range(start_t, start_t + num_frames)}
    for src_id, tgt_id in tracking_result.edges:
        if src_id in node_dict and tgt_id in node_dict:
            src_node = node_dict[src_id]
            if start_t <= src_node.t < start_t + num_frames:
                t_edges[src_node.t].append((src_node, node_dict[tgt_id]))
                
    for idx, t in enumerate(range(start_t, start_t + num_frames)):
        if t >= len(raw_frames):
            break
        ax = axes[idx]
        mip = np.max(raw_frames[t], axis=0)
        ax.imshow(mip, cmap="gray", origin="upper")
        
        t_nodes = [n for n in tracking_result.nodes if n.t == t]
        ax.scatter([n.x for n in t_nodes], [n.y for n in t_nodes], color="cyan", s=35, zorder=3, label="Cells")
        
        edges = t_edges.get(t, [])
        for src, tgt in edges:
            ax.annotate(
                "",
                xy=(tgt.x, tgt.y),
                xytext=(src.x, src.y),
                arrowprops=dict(arrowstyle="->", color="lime", lw=1.8, alpha=0.95),
            )
            
        ax.set_title(f"Timepoint t={t} ({len(t_nodes)} cells, {len(edges)} links)", fontsize=13, fontweight="bold")
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        
    plt.tight_layout()
    plt.show()


In [ ]:
# Run Inference & Display Overlay Predictions

# ============================================================
# USER SETTINGS -- change threshold here
DETECTION_THRESHOLD = 0.4   # only show/count predictions >= this confidence
# ============================================================

import glob, os, json
search_dirs = ["/kaggle/input/competitions/biohub-cell-tracking-during-development/train"]
samples = []
for s_dir in search_dirs:
    samples.extend(glob.glob(f"{s_dir}/*.zarr"))
    samples.extend(glob.glob(f"{s_dir}/*/*.zarr"))
samples = sorted(list(set(samples)))

if samples:
    target_sample = samples[0]
    print(f"Processing: {target_sample}")

    # Load ground-truth GEFF
    gt_nodes_t0 = []
    try:
        from src.data.geff_loader import load_geff, geff_path_for_zarr
        from pathlib import Path
        geff_p = geff_path_for_zarr(target_sample)
        if Path(geff_p).exists():
            gt_graph = load_geff(geff_p)
            all_t_vals = sorted({n.t for n in gt_graph.nodes})
            gt_nodes_t0 = gt_graph.nodes_at(0)
            # If t=0 is empty, use the first available timepoint
            if not gt_nodes_t0 and all_t_vals:
                first_t = all_t_vals[0]
                gt_nodes_t0 = gt_graph.nodes_at(first_t)
                print(f"[GT]  t=0 has no GT — using t={first_t} instead.")
            print(f"[GT]  Cells in ground truth at t=0 (across all Z): {len(gt_nodes_t0)}")
            print(f"[GT]  Total GT nodes all timepoints: {len(gt_graph.nodes)}")
            print(f"[GT]  All timepoints in GEFF: {all_t_vals}")
            # Z range of GT
            if gt_nodes_t0:
                zs = [n.z for n in gt_nodes_t0]
                print(f"[GT]  Z range at t=0: min={min(zs)}, max={max(zs)}, unique={len(set(zs))} slices")
        else:
            print(f"[GT]  No .geff file found at {geff_p}")
    except Exception as e:
        print(f"[GT]  Could not load ground truth: {e}")

    # Run inference
    raw_frames, frame_detections, tracking_result = run_sample_inference(
        target_sample, model, config, timepoints_to_process=1
    )

    # Summary
    pred_above = [n for n in frame_detections[0] if n.confidence >= DETECTION_THRESHOLD]
    pred_zs = [int(round(n.z)) for n in pred_above]
    print(f"\n[PRED] Total peaks extracted (all Z, conf>=0.05): {len(frame_detections[0])}")
    print(f"[PRED] Cells shown (conf>={DETECTION_THRESHOLD}): {len(pred_above)}")
    if pred_above:
        print(f"[PRED] Z range: min={min(pred_zs)}, max={max(pred_zs)}, unique={len(set(pred_zs))} slices")
    print(f"[GT]   GT cells at t=0: {len(gt_nodes_t0)}")

    # Visualize
    visualize_with_gt(
        raw_frames[0], frame_detections[0], gt_nodes_t0,
        timepoint=0, threshold=DETECTION_THRESHOLD
    )
else:
    print("No .zarr samples found.")


Processing sample volume: /kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_0113de3b.zarr
